# Урок 5. Мини-исследование: спортивный аналитик АПЛ

**Имя и фамилия:**  
**Группа:**  
**Номера назначенных команд:**


## Как работать

Структура вашей работы в репозитории:
python-ai-course/
└── homework/
    └── lesson-05/
        └── solution.ipynb

1. Откройте или сохраните этот ноутбук как homework/lesson-05/solution.ipynb.
2. Запускайте ячейки сверху вниз. Если CSV ещё не загружен в текущий сеанс Colab, появится выбор файла. Выберите выданный преподавателем premier_league_2025_26.csv — можно найти его в той же папке на компьютере, где лежит ноутбук. Colab скопирует файл во временную папку /content. После перезапуска сеанса его может понадобиться выбрать снова: /content не является папкой Google Диска и очищается вместе с сеансом.
3. В ячейку с номерами внесите ровно пять назначенных вам номеров через запятую, например [2, 7, 11, 15, 19].
4. В отмеченных Code-ячейках дописывайте Python-код. Первые готовые примеры можно копировать и менять под следующий вопрос.
5. После вычислений запишите вывод под каждой задачей. Подкрепляйте каждое утверждение числами и матчами из результатов ячеек. Не делайте вывод только по просмотру исходной таблицы.
6. Во время занятия сохраните промежуточную версию отдельным коммитом. Дома завершите ноутбук, запустите Restart Kernel → Run All, исправьте ошибки и сохраните итог отдельным коммитом.

Отдельная папка с результатами не создаётся. Справочник команд, ваши назначения и результаты кода показываются под соответствующими ячейками. В репозитории сохраняется сам ноутбук homework/lesson-05/solution.ipynb; загруженный CSV остаётся во временной среде Colab.


## Исследовательская задача

Вы — спортивный аналитик. Исследуйте пять назначенных команд, сравнивайте показатели и обосновывайте выводы данными.

Преподаватель заранее проверил набор: в нём 380 матчей. Искать пропуски и проверять полноту не нужно. Каждая строка — матч. HomeTeam — хозяева, AwayTeam — гости. Например, в Liverpool — Bournemouth со счётом 4:2 команда Liverpool забила 4 гола, а Bournemouth — 2. В таблице это FTHG и FTAG; FTR="H" означает победу хозяев, FTR="A" — победу гостей, D — ничью.

Чтобы сравнивать команды одинаково, подготовленная ячейка создаёт запись с позиции каждой команды: goals_for — её голы, goals_against — голы соперника, *_for и *_against — её показатели и показатели соперника. Для гостя значения уже переведены в его перспективу. Запустите ячейку и используйте её результаты; вручную «переворачивать» матч не нужно.

У каждого этапа есть пример для понимания типа анализа. Это иллюстрация, а не единственный правильный ответ: можно выбрать другой показатель, пример и способ расчёта, если код соответствует вопросу, а вывод подтверждён числами.


| Столбец | Что означает |
|---|---|
| `Date` | дата матча |
| `HomeTeam` | команда хозяев, играет на своём поле |
| `AwayTeam` | команда гостей, играет на поле хозяев |
| `FTHG` | голы хозяев за весь матч |
| `FTAG` | голы гостей за весь матч |
| `FTR` | итог: `H` — победили хозяева, `D` — ничья, `A` — победили гости |
| `HS` / `AS` | все удары хозяев / гостей |
| `HST` / `AST` | удары в створ ворот хозяев / гостей |
| `HC` / `AC` | угловые хозяев / гостей |
| `HF` / `AF` | фолы хозяев / гостей |
| `HY` / `AY` | жёлтые карточки хозяев / гостей |
| `HR` / `AR` | красные карточки хозяев / гостей |

## Загрузка набора данных

Если premier_league_2025_26.csv уже находится в текущей среде Colab (/content), ячейка прочитает его сразу. Иначе откроется выбор файла: выберите CSV с компьютера. Даже если файл лежит рядом с ноутбуком в Google Диске, Colab не помещает его автоматически в /content; его нужно выбрать для текущего сеанса.

После загрузки ячейка покажет число матчей и команд. Список команд с номерами появится в выводе ячейки «Этап 1» — никакая отдельная папка или файл для списка не создаётся.


In [ ]:
from pathlib import Path
import csv

file_name = "premier_league_2025_26.csv"
data_path = Path("/content") / file_name

if not data_path.exists():
    from google.colab import files
    print("Выберите premier_league_2025_26.csv с компьютера. Файл будет помещён во временную среду Colab /content.")
    uploaded_files = files.upload()
    if file_name not in uploaded_files:
        raise FileNotFoundError("Выберите файл premier_league_2025_26.csv")

numeric_fields = ["FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF", "HY", "AY", "HR", "AR"]
with data_path.open(encoding="utf-8-sig", newline="") as f:
    matches = list(csv.DictReader(f))
for match in matches:
    for field in numeric_fields:
        match[field] = int(match[field])
all_teams = sorted({m["HomeTeam"] for m in matches} | {m["AwayTeam"] for m in matches}, key=str.casefold)
team_catalog = {n: name for n, name in enumerate(all_teams, start=1)}
print(f"Загружено матчей: {len(matches)}; команд: {len(team_catalog)}")


## Этап 1. Справочник команд и ваши номера

После загрузки файла ноутбук собирает все уникальные названия из столбцов хозяев и гостей, сортирует их по алфавиту и присваивает номера с 1. Это справочник `team_catalog`: номер → название команды. Он создаётся в памяти текущего сеанса Colab, а не сохраняется в отдельной папке или файле.

Справочник будет показан в выводе ячейки ниже. Преподаватель сообщит вам пять назначенных номеров. Впишите их **через запятую**, например `[2, 7, 11, 15, 19]`. После запуска ячейка покажет названия ваших команд; они будут доступны ноутбуку в списке `selected_team_names`.

In [ ]:
assigned_team_numbers = [19, 16, 8, 4, 10]  # Мои назначенные номера команд
selected_team_names = []
if len(assigned_team_numbers) == 5 and len(set(assigned_team_numbers)) == 5 and all(n in team_catalog for n in assigned_team_numbers):
    selected_team_names = [team_catalog[n] for n in assigned_team_numbers]
    print("Ваши команды:", selected_team_names)
else:
    print("Впишите ровно пять разных номеров через запятую.")
print("Справочник команд — номер и название:")
for number, name in team_catalog.items():
    print(f"{number:>2}. {name}")

## Этап 2. Подготовленные записи матчей

Хозяева играют на своём поле и записаны первыми; гости — на поле хозяев и записаны вторыми. Например, в матче Liverpool — Bournemouth 4:2 у Liverpool забитые голы — 4, у Bournemouth — 2.

Следующая ячейка уже готова: она для всех 380 матчей создаёт записи обеих команд. В каждой записи показатели собраны одинаково: `goals_for` — голы этой команды, `goals_against` — голы соперника; `*_for` и `*_against` так же обозначают её показатели и показатели соперника. Результат `W/D/L` определяется по счёту. Вам не нужно писать или менять эту ячейку: запустите её и дальше анализируйте `selected_team_games` — матчи пяти назначенных команд.

In [ ]:
team_games = []

for match in matches:
    sides = [
        {
            "team": match["HomeTeam"], "opponent": match["AwayTeam"], "venue": "дома",
            "goals_for": match["FTHG"], "goals_against": match["FTAG"],
            "shots_for": match["HS"], "shots_against": match["AS"],
            "target_for": match["HST"], "target_against": match["AST"],
            "corners_for": match["HC"], "corners_against": match["AC"],
            "fouls_for": match["HF"], "fouls_against": match["AF"],
            "yellow_for": match["HY"], "yellow_against": match["AY"],
            "red_for": match["HR"], "red_against": match["AR"],
            "win_code": "H"
        },
        {
            "team": match["AwayTeam"], "opponent": match["HomeTeam"], "venue": "в гостях",
            "goals_for": match["FTAG"], "goals_against": match["FTHG"],
            "shots_for": match["AS"], "shots_against": match["HS"],
            "target_for": match["AST"], "target_against": match["HST"],
            "corners_for": match["AC"], "corners_against": match["HC"],
            "fouls_for": match["AF"], "fouls_against": match["HF"],
            "yellow_for": match["AY"], "yellow_against": match["HY"],
            "red_for": match["AR"], "red_against": match["HR"],
            "win_code": "A"
        }
    ]

    for side in sides:
        if side["goals_for"] > side["goals_against"]:
            result = "W"
        elif side["goals_for"] == side["goals_against"]:
            result = "D"
        else:
            result = "L"
        side["date"] = match["Date"]
        side["result"] = result
        side["points"] = 3 if result == "W" else 1 if result == "D" else 0
        del side["win_code"]
        team_games.append(side)

selected_team_games = []
for game in team_games:
    if game["team"] in selected_team_names:
        selected_team_games.append(game)

print("Записей по всем командам:", len(team_games))
print("Записей по вашим пяти командам:", len(selected_team_games))
for name in selected_team_names:
    games_count = 0
    for game in selected_team_games:
        if game["team"] == name:
            games_count += 1
    print(name, "матчей:", games_count)

## Этап 3. Профиль сезона

**Вопросы 1–2.** Для каждой из пяти команд посчитайте победы, ничьи, поражения, забитые и пропущенные голы, разницу голов и очки (победа — 3, ничья — 1, поражение — 0). Кто лидер среди ваших команд по очкам? Совпадает ли он с лидером по разнице голов?

Ниже дан готовый пример сводки. Запустите его и посмотрите, как список матчей превращается в итоги сезона. В следующих этапах можно скопировать цикл и изменить показатели.

**Пример формата вывода:** в этом наборе Arsenal закончил сезон с 85 очками и разницей голов +44, а Man City — с 78 очками и разницей +42. Для своей пятёрки сравните рассчитанные строки. Это пример чтения сводки; возможны другие способы расчёта и оформления.


In [ ]:
team_profiles = []

for team in selected_team_names:
    wins = 0
    draws = 0
    losses = 0
    goals_for = 0
    goals_against = 0

    for game in selected_team_games:
        if game["team"] == team:
            goals_for += game["goals_for"]
            goals_against += game["goals_against"]
            if game["result"] == "W":
                wins += 1
            elif game["result"] == "D":
                draws += 1
            else:
                losses += 1

    goal_difference = goals_for - goals_against
    points = wins * 3 + draws
    team_profiles.append({
        "team": team, "wins": wins, "draws": draws, "losses": losses,
        "goals_for": goals_for, "goals_against": goals_against,
        "goal_difference": goal_difference, "points": points
    })

for profile in team_profiles:
    print(profile)

### ? ВЫВОДЫ

Назовите лидера по очкам и по разнице голов и подтвердите числами.

**Ответ:**

Среди моих пяти команд лидер по очкам — **Brentford: 53 очка**. По разнице голов лучший результат тоже у **Brentford: +3**. Поэтому лидер по очкам и лидер по разнице голов совпадают.

## Этап 4. Домашние и выездные матчи

**Вопрос 3.** Для каждой команды сравните число побед и среднее количество забитых голов дома и в гостях. Для каждого среднего укажите число матчей, на котором оно рассчитано.

**Пример:** в наборе Man City забивал в среднем 2,37 гола дома и 1,68 в гостях (по 19 матчей). Покажите такой же расчёт для своих команд. Среднее можно получить суммой голов, поделённой на число матчей; допустим и другой понятный способ.


In [ ]:
home_away_summary = []

for team in selected_team_names:
    home_wins = 0
    away_wins = 0
    home_goals = 0
    away_goals = 0
    home_games = 0
    away_games = 0

    for game in selected_team_games:
        if game["team"] == team:
            if game["venue"] == "дома":
                home_games += 1
                home_goals += game["goals_for"]
                if game["result"] == "W":
                    home_wins += 1
            else:
                away_games += 1
                away_goals += game["goals_for"]
                if game["result"] == "W":
                    away_wins += 1

    home_average = home_goals / home_games
    away_average = away_goals / away_games

    home_away_summary.append({
        "team": team,
        "home_wins": home_wins,
        "away_wins": away_wins,
        "home_average_goals": round(home_average, 2),
        "home_games": home_games,
        "away_average_goals": round(away_average, 2),
        "away_games": away_games
    })

for summary in home_away_summary:
    print(summary)

### ? ВЫВОДЫ

У какой команды заметнее разница между домашней и выездной игрой? Приведите числа.

**Ответ:**

Самая заметная разница у **West Ham**. Дома команда выиграла 6 матчей и забила 27 голов в 15 матчах, то есть в среднем **1,80 гола за матч**. В гостях — 4 победы и 19 голов в 19 матчах, то есть **1,00 гола за матч**. Разница средней результативности составляет **0,80 гола за матч**.

## Этап 5. Проверка гипотезы

**Вопросы 4–5.** Сравните выбранные команды по голам и ударам в створ. Проверьте гипотезу: «Преимущество по ударам в створ связано с победой». Для каждой команды посчитайте матчи, где она нанесла больше ударов в створ, число побед в них и отдельно матчи с равным числом ударов в створ.

**Пример контрпримера:** 24.11.2025 Everton победил Man United 1:0, хотя уступил по ударам в створ 1:6 и по общему числу ударов 3:25. Этот матч показывает, почему связь не означает, что преимущество гарантирует победу. Можно привести другой матч из результатов своей ячейки.


In [ ]:
target_comparison = []

for team in selected_team_names:
    more_target = 0
    wins_when_more = 0
    equal_target = 0

    for game in selected_team_games:
        if game["team"] == team:
            if game["target_for"] > game["target_against"]:
                more_target += 1
                if game["result"] == "W":
                    wins_when_more += 1
            elif game["target_for"] == game["target_against"]:
                equal_target += 1

    target_comparison.append({
        "team": team,
        "more_target_matches": more_target,
        "wins_when_more": wins_when_more,
        "equal_target_matches": equal_target
    })

for result in target_comparison:
    print(result)

### ? ВЫВОДЫ

Поддерживают ли результаты гипотезу? Приведите пример за и контрпример.

**Ответ:**

Результаты **частично поддерживают** гипотезу, потому что есть матчи, где преимущество по ударам в створ сопровождалось победой. Например, **09.11.2025 Brentford — Newcastle 3:1**: Brentford нанёс 7 ударов в створ против 1 и выиграл.

Но преимущество по ударам в створ не гарантирует победу. Например, **07.01.2026 Fulham — Chelsea 2:1**: Fulham победил, хотя уступил по ударам в створ **5:6**. Поэтому по этим данным связь есть, но она не является правилом для каждого матча.

## Этап 6. Матч-детектив

**Вопрос 6.** Найдите победу при меньшем числе ударов в створ или поражение при большем. Укажите дату, команды, счёт и удары в створ с обеих сторон.

**Вопрос 7.** У какой из ваших пяти команд была самая крупная победа? Найдите матч, укажите дату, соперников и счёт, затем посчитайте разницу забитых и пропущенных голов. Если у нескольких матчей одинаковый максимум, приведите их все.

**Пример по всему набору:** максимальная разница в счёте — 5 голов. Например, 23.08.2025 Arsenal победил Leeds 5:0; ещё один матч с такой разницей — Sunderland — Nott'm Forest 0:5 (24.04.2026). Для ответа используйте только матчи ваших назначенных команд.

**На выбор — выполните один дополнительный пункт:**
- Найдите матч одной из ваших команд, в котором забили больше всего голов суммарно. Укажите счёт и сравните его с обычным результатом команды.
- Какая из ваших команд чаще всего не пропускала? Посчитайте такие матчи и приведите пример.

Пример для дополнительного варианта с максимумом голов: 02.12.2025 Fulham — Man City 4:5, всего 9 голов. Для «сухих» матчей, например, можно найти среди результатов Arsenal — Leeds 5:0 (23.08.2025). Это только иллюстрации; допустим другой пример и другой способ поиска, если код и вывод отвечают на вопрос.


In [ ]:
surprising_matches = []

for game in selected_team_games:
    if game["result"] == "W" and game["target_for"] < game["target_against"]:
        surprising_matches.append({
            "date": game["date"],
            "team": game["team"],
            "opponent": game["opponent"],
            "score": str(game["goals_for"]) + ":" + str(game["goals_against"]),
            "target_for": game["target_for"],
            "target_against": game["target_against"]
        })
    elif game["result"] == "L" and game["target_for"] > game["target_against"]:
        surprising_matches.append({
            "date": game["date"],
            "team": game["team"],
            "opponent": game["opponent"],
            "score": str(game["goals_for"]) + ":" + str(game["goals_against"]),
            "target_for": game["target_for"],
            "target_against": game["target_against"]
        })

print("Неожиданные матчи:")
for match in surprising_matches:
    print(match)

In [ ]:
largest_wins = []
max_difference = -1

for game in selected_team_games:
    if game["result"] == "W":
        difference = game["goals_for"] - game["goals_against"]
        if difference > max_difference:
            max_difference = difference

for game in selected_team_games:
    if game["result"] == "W":
        difference = game["goals_for"] - game["goals_against"]
        if difference == max_difference:
            largest_wins.append({
                "date": game["date"],
                "team": game["team"],
                "opponent": game["opponent"],
                "score": str(game["goals_for"]) + ":" + str(game["goals_against"]),
                "goal_difference": difference
            })

print("Самые крупные победы:")
for match in largest_wins:
    print(match)

### ? ВЫВОДЫ

Объясните, что показывает найденный матч и чего он не доказывает. Для победы с максимальной разницей приведите счёт и расчёт.

**Ответ:**

Самая крупная победа среди моих команд — **24.04.2026 Sunderland — Nott'm Forest 0:5**. Разница забитых и пропущенных голов: **5 − 0 = 5**.

Этот матч показывает, что Nott'm Forest смогла выиграть с очень большим преимуществом по счёту. Но один матч не доказывает, что такая разница будет повторяться в других играх или что она была вызвана только одним показателем.

## Этап 7. Угловые и карточки

**Вопросы 8–9.** У какой из ваших команд больше угловых в среднем? Сопоставьте угловые с очками или победами. Посчитайте жёлтые и красные карточки отдельно. Всегда ли команда с большим числом карточек проигрывала?

**Пример неожиданного матча:** 23.08.2025 Man City проиграл Tottenham 0:2, хотя подал больше угловых — 7 против 2. Для карточек можно найти собственный пример и отдельно указать жёлтые и красные. Не утверждайте, что угловые или карточки сами по себе вызвали результат.


In [ ]:
corner_card_summary = []

for team in selected_team_names:
    games_count = 0
    corners_for = 0
    wins = 0
    points = 0
    yellow = 0
    red = 0

    for game in selected_team_games:
        if game["team"] == team:
            games_count += 1
            corners_for += game["corners_for"]
            yellow += game["yellow_for"]
            red += game["red_for"]
            wins += 1 if game["result"] == "W" else 0
            points += game["points"]

    corner_card_summary.append({
        "team": team,
        "average_corners": round(corners_for / games_count, 2),
        "wins": wins,
        "points": points,
        "yellow_cards": yellow,
        "red_cards": red
    })

for result in corner_card_summary:
    print(result)

print("\nМатчи, где команда имела больше угловых, но проиграла:")
for game in selected_team_games:
    if game["corners_for"] > game["corners_against"] and game["result"] == "L":
        print({
            "date": game["date"],
            "team": game["team"],
            "opponent": game["opponent"],
            "score": str(game["goals_for"]) + ":" + str(game["goals_against"]),
            "corners_for": game["corners_for"],
            "corners_against": game["corners_against"]
        })

print("\nМатчи, где команда имела больше карточек и не проиграла:")
for game in selected_team_games:
    cards_for = game["yellow_for"] + game["red_for"]
    cards_against = game["yellow_against"] + game["red_against"]
    if cards_for > cards_against and game["result"] != "L":
        print({
            "date": game["date"],
            "team": game["team"],
            "opponent": game["opponent"],
            "score": str(game["goals_for"]) + ":" + str(game["goals_against"]),
            "yellow_for": game["yellow_for"],
            "red_for": game["red_for"],
            "yellow_against": game["yellow_against"],
            "red_against": game["red_against"]
        })

### ? ВЫВОДЫ

Приведите один пример за или против связи показателя с результатом. Не утверждайте причинность.

**Ответ:**

Угловые не всегда совпадали с результатом матча. Например, **23.08.2025 Man City — Tottenham 0:2**: Man City подал **7 угловых**, а Tottenham — **2**, но победил Tottenham. Значит, большее количество угловых само по себе не гарантирует победу.

Карточки тоже не стоит считать причиной результата. В анализе нужно отдельно смотреть количество жёлтых и красных карточек и итог матча.

## Этап 8. Сравнение и вывод аналитика

**Вопросы 10–13.** Выберите две из своих пяти команд с похожим числом очков или побед и сравните их по двум другим показателям. Сформулируйте три вывода по всем пяти командам; каждый подтвердите результатом кода. Добавьте один неожиданный результат и вопрос, на который этот набор данных не отвечает.

**Пример выбора:** можно сравнить команды с близкими очками, но разным числом забитых голов или разницей голов. Например, Arsenal (85 очков) и Man City (78) — близкие лидеры сезона, но этот пример не подменяет выбор команд из вашего задания. Подойдут и другие команды с другими показателями.


In [ ]:
compare_team_numbers = [4, 10]  # Brentford и Fulham — близкие по очкам команды
comparison = []

compare_team_names = [team_catalog[number] for number in compare_team_numbers]

for profile in team_profiles:
    if profile["team"] in compare_team_names:
        comparison.append(profile)

print("Сравнение команд по итогам сезона:")
for item in comparison:
    print(item)

print("\nДополнительное сравнение:")
for team in compare_team_names:
    for summary in home_away_summary:
        if summary["team"] == team:
            print({
                "team": team,
                "home_wins": summary["home_wins"],
                "away_wins": summary["away_wins"],
                "home_average_goals": summary["home_average_goals"],
                "away_average_goals": summary["away_average_goals"]
            })

for item in corner_card_summary:
    if item["team"] in compare_team_names:
        print({
            "team": item["team"],
            "average_corners": item["average_corners"],
            "yellow_cards": item["yellow_cards"],
            "red_cards": item["red_cards"]
        })

### ? ВЫВОДЫ — итог мини-исследования

**Три вывода с доказательствами:**

1. **Brentford показал лучший результат среди моих пяти команд по очкам:** 53 очка и разница голов +3. Это выше, чем у Fulham (52), Crystal Palace (45), Nott'm Forest (44) и West Ham (39).
2. **West Ham заметно сильнее играл дома по результативности:** 27 голов в 15 домашних матчах (1,80 за матч) против 19 голов в 19 выездных матчах (1,00 за матч).
3. **Преимущество по ударам в створ не гарантирует победу.** Например, 07.01.2026 Fulham победил Chelsea 2:1, хотя уступил по ударам в створ 5:6.

**Неожиданный матч или результат:**

24.04.2026 Sunderland — Nott'm Forest 0:5. Nott'm Forest забила пять голов и выиграла с разницей +5.

**Вопрос, на который данных не хватает:**

По этим данным нельзя точно определить, **почему** команда выиграла конкретный матч. В таблице есть счёт, удары, угловые, фолы и карточки, но нет, например, информации о качестве моментов, тактике, травмах и других причинах результата.

## Коммиты и домашнее завершение

Сохраните ноутбук в том же файле:

`homework/lesson-05/solution.ipynb`

После завершения выполните **Restart Kernel → Run All**, проверьте, что все ячейки выполняются сверху вниз без ошибок, и сохраните итог отдельным коммитом:

**HW-05 — итоговая версия**

**Срок доработки: 21 октября 2026 года, 23:59 по Москве.**